# Predicting California house prices

`explore.ipynb` got to know the data. This notebook starts building a **model**: a program that learns the relationship between a neighbourhood's details (income, location, house age and so on) and its house price from examples, so that it can predict the price for neighbourhoods it has not seen.

Before any model can be trained, the data has to be prepared in three steps:

1. Load the data.
2. Separate the columns the model learns from (the **features**) from the column it predicts (the **target**).
3. Set aside part of the rows as a **test set** for checking the model later.

Section 4 then sets a **baseline**: the error of the simplest possible prediction, which every real model will have to beat.

Section 5 trains the first real model, a **linear regression**, and compares its error with the baseline.

Section 6 trains a more flexible model, a **random forest**, and section 7 puts all three results side by side in one table.

Section 8 compares each model's error on the training set with its error on the test set to check for **overfitting**.

Section 9 uses the random forest to predict the price for a made-up neighbourhood whose details are typed in by hand.

## 1. Load the data

This is the same loading step as in `explore.ipynb`. `fetch_california_housing(as_frame=True)` returns the dataset with the table stored as a pandas **DataFrame** (a table with named columns) in `data.frame`.

All the imports for this notebook live in this first code cell. `train_test_split` is not used until section 3, `numpy` and `mean_absolute_error` not until section 4, `LinearRegression` not until section 5, `RandomForestRegressor` not until section 6 and `pandas` not until section 7, but keeping imports together at the top makes it easy to see what the notebook depends on.

**What to look for:** the shape should be `(20640, 9)`, meaning 20,640 rows (neighbourhoods) and 9 columns (8 features plus the price).

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

data = fetch_california_housing(as_frame=True)
df = data.frame

print("Shape (rows, columns):", df.shape)
df.head()

Shape (rows, columns): (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


## 2. Split the columns into features X and target y

A model learns to predict one column from the others, so the table is cut into two pieces by column:

- **Features** are the inputs: the 8 columns the model is allowed to look at (`MedInc`, `HouseAge`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude`, `Longitude`).
- The **target** is the answer the model tries to predict: `MedHouseVal`, the median house value in units of $100,000.

By convention the features are called `X` (capital, because it is a table with many columns) and the target is called `y` (lowercase, because it is a single column). This comes from maths, where `y = f(X)` means "y depends on X". Almost all scikit-learn code uses these names, so it is worth getting used to them.

`df.drop(columns="MedHouseVal")` returns a copy of the table without the price column. It does not change `df` itself. Removing the target from `X` matters: if the price were left in the features, the model could simply copy it and would learn nothing.

**What to look for:** `X` should have 8 columns and `y` should have none (its shape is `(20640,)`, a single column of 20,640 values). Both must have the same number of rows, because row 0 of `X` and row 0 of `y` describe the same neighbourhood.

In [2]:
X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Feature columns:", list(X.columns))

X shape: (20640, 8)
y shape: (20640,)
Feature columns: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']


## 3. Split the rows into a training set and a test set

The second split is by row. 80% of the neighbourhoods become the **training set**, which the model learns from. The other 20% become the **test set**, which is locked away and only used at the end to check how good the model is.

### Why keep a test set the model never sees?

The goal is not a model that does well on these 20,640 neighbourhoods. Their prices are already known. The goal is a model that does well on *new* neighbourhoods, and the only way to measure that is to test it on data it has never seen.

Think of a student preparing for an exam. If the exam uses exactly the questions from the practice sheet, a student who memorised the answers scores 100% without understanding anything. Only fresh questions show whether they learned the subject. A model can do the same thing: it can memorise the quirks of its training rows instead of learning the general pattern. This is called **overfitting**. An overfitted model looks excellent on the data it trained on and then does badly in real use.

The test set is the fresh exam. Because the model never saw those rows during training, its score on them is an honest estimate of how it will perform on new data. If the training score is much better than the test score, that gap is the warning sign of overfitting.

This only works if the test set stays truly unseen. If test rows influence any decision made while building the model (which features to use, which settings to pick), information leaks from the exam into the revision, and the final score becomes too optimistic. This mistake is called **data leakage**. The rule is: split first, then do everything else using the training set only, and look at the test set once at the end.

### What the code does

`train_test_split` shuffles the rows and then cuts them. It splits `X` and `y` together so that each neighbourhood's features stay matched with its price.

- `test_size=0.2` puts 20% of the rows in the test set. 80/20 is a common choice: most of the data goes to learning, and about 4,000 rows is still plenty for a reliable score.
- `random_state=42` fixes the shuffle so the same rows land in each set every time the notebook runs. Without it the split would change on each run and so would the results, which makes work impossible to reproduce. The number itself is arbitrary. Any fixed integer does the job.
- The function returns four pieces, always in this order: `X_train, X_test, y_train, y_test`.

**What to look for:** 16,512 training rows and 4,128 test rows, which add up to 20,640. The two average prices should be close to each other, which shows that the shuffle gave both sets a similar mix of cheap and expensive neighbourhoods.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set:", X_train.shape, y_train.shape)
print("Test set:    ", X_test.shape, y_test.shape)
print(f"Share of rows in the test set: {len(X_test) / len(X):.0%}")
print(f"Average price in training set: {y_train.mean():.3f} ($100,000s)")
print(f"Average price in test set:     {y_test.mean():.3f} ($100,000s)")

Training set: (16512, 8) (16512,)
Test set:     (4128, 8) (4128,)
Share of rows in the test set: 20%
Average price in training set: 2.072 ($100,000s)
Average price in test set:     2.055 ($100,000s)


## 4. How wrong is the simplest possible prediction?

Before training a real model, it helps to know what score a model with no intelligence at all would get. That score is called a **baseline**. Without one, an error number means nothing on its own: there is no way to tell whether a model that is off by $50,000 is good or bad until it is compared with something.

The baseline here ignores all 8 features and predicts the same price for every neighbourhood: the average price of the training set. A real model only earns its keep if it beats this.

### Measuring the error with MAE

**Mean absolute error (MAE)** is calculated in three steps:

1. For each neighbourhood, take the **error**: the actual price minus the predicted price.
2. Take the **absolute value** of each error, which drops the minus sign. Being $50,000 too high and $50,000 too low both count as $50,000 wrong. Without this step, errors in opposite directions would cancel out and a bad model could look perfect.
3. Take the **mean** (average) of those values.

The result answers one question: on average, how far off is a prediction?

### What MAE means in dollars

MAE is in the same units as the target, which is its main advantage. `MedHouseVal` is in units of $100,000, so multiplying the MAE by 100,000 gives dollars. An MAE of 0.5 means predictions miss by $50,000 on average.

Three things to keep in mind when reading it:

- It is an average, not a guarantee. Some predictions will be much closer and some much further off.
- Each row is a neighbourhood, so the error is in the neighbourhood's *median* house value, not the price of one house.
- These are 1990 prices, and values above $500,000 were recorded as $500,001 (the cap found in `explore.ipynb`). For those neighbourhoods the true price is unknown, so the true error is at least as large as the one measured.

### What the code does

- `y_train.mean()` computes the average from the **training set only**. Using the test set's average would be the data leakage described in section 3, even for a model this simple.
- `np.full(len(y_test), baseline_price)` builds an array holding that one number repeated 4,128 times, one prediction for each test row.
- `mean_absolute_error(y_test, baseline_predictions)` compares the actual test prices with the predictions. The actual values go first by scikit-learn convention.

**What to look for:** the predicted price is the 2.072 training average from section 3, and the MAE should be about 0.906.

**Finding:** predicting the average price for every neighbourhood is wrong by about $90,600 on average. The typical neighbourhood is worth about $207,000, so that is an error of roughly 44% of the price. Any model trained from here on needs an MAE below 0.906 to be worth using.

In [4]:
DOLLARS_PER_UNIT = 100_000  # MedHouseVal is in units of $100,000

# The baseline "learns" one number from the training set: the average price.
baseline_price = y_train.mean()

# Predict that same price for every neighbourhood in the test set.
baseline_predictions = np.full(len(y_test), baseline_price)

baseline_mae = mean_absolute_error(y_test, baseline_predictions)

print(f"Baseline prediction for every neighbourhood: {baseline_price:.3f} "
      f"(${baseline_price * DOLLARS_PER_UNIT:,.0f})")
print(f"Baseline MAE on the test set: {baseline_mae:.3f} "
      f"(${baseline_mae * DOLLARS_PER_UNIT:,.0f})")
print(f"MAE as a share of the average price: {baseline_mae / baseline_price:.0%}")

Baseline prediction for every neighbourhood: 2.072 ($207,195)
Baseline MAE on the test set: 0.906 ($90,607)
MAE as a share of the average price: 44%


## 5. Does a linear regression beat the baseline?

The baseline ignored every feature. **Linear regression** is the simplest model that uses them.

### How linear regression works: the taxi fare

A taxi fare is worked out from a fixed rule: a starting charge as soon as you get in, plus a set rate for every kilometre, plus a set rate for every minute. Once you know the three numbers, you can price any trip.

Linear regression prices a neighbourhood the same way. It has a starting price, plus one rate for each of the 8 features: so much added for every extra $10,000 of median income, so much added or taken away for every extra year of house age, and so on. Adding up the starting price and the 8 amounts gives the prediction.

Nobody tells the model what the rates are. **Training** (also called **fitting**) is the model working them out for itself. It is like being handed 16,512 old taxi receipts with the distance, the time and the final fare, and adjusting the rates until the rule reproduces those fares as closely as possible. Here the receipts are the training neighbourhoods and the fares are their known prices.

The rates never change, which is both the strength and the weakness of this model. A taxi that charges the same per kilometre on the first kilometre and on the hundredth is easy to understand, but it cannot offer a long-trip discount. In the same way, linear regression assumes each extra $10,000 of income adds the same amount to the price whether the neighbourhood is poor or rich. On a chart that fixed rate is a straight line, which is where the name **linear** comes from.

### What the code does

scikit-learn models all follow the same three steps, so this pattern will come back with every model:

- `LinearRegression()` creates an empty model that knows nothing yet.
- `model.fit(X_train, y_train)` trains it: it works out the starting price and the 8 rates from the **training set only**.
- `model.predict(X_test)` applies those rates to the test neighbourhoods, which it has never seen, and returns one predicted price for each.

`mean_absolute_error` then scores the predictions exactly as in section 4, so the two numbers can be compared fairly. The code also measures the MAE on the training set to check for the overfitting described in section 3.

**What to look for:** a test MAE of about 0.533, against 0.906 for the baseline. The training MAE (about 0.529) should be close to the test MAE, which means the model has not overfitted. The lowest predicted price is below zero, which is impossible for a house.

**Finding:** linear regression is wrong by about $53,300 on average, compared with $90,600 for the baseline. That is $37,300 closer, a 41% lower error, so the 8 features do carry real information about price. The model is still off by about 26% of a typical neighbourhood's price, and its fixed rates produce a few impossible predictions (negative prices, and prices far above the $500,000 cap), so there is room for a more flexible model to do better.

In [5]:
model = LinearRegression()

# Learn the starting price and the 8 rates from the training set only.
model.fit(X_train, y_train)

# Predict a price for every neighbourhood in the test set.
linear_predictions = model.predict(X_test)

linear_mae = mean_absolute_error(y_test, linear_predictions)
train_mae = mean_absolute_error(y_train, model.predict(X_train))
improvement = baseline_mae - linear_mae

print(f"Baseline MAE on the test set:          {baseline_mae:.3f} "
      f"(${baseline_mae * DOLLARS_PER_UNIT:,.0f})")
print(f"Linear regression MAE on the test set: {linear_mae:.3f} "
      f"(${linear_mae * DOLLARS_PER_UNIT:,.0f})")
print(f"Improvement over the baseline:         {improvement:.3f} "
      f"(${improvement * DOLLARS_PER_UNIT:,.0f}, "
      f"{improvement / baseline_mae:.0%} lower)")
print(f"Linear regression MAE on the training set: {train_mae:.3f}")
print(f"Lowest and highest predicted price: {linear_predictions.min():.2f} "
      f"and {linear_predictions.max():.2f} ($100,000s)")

Baseline MAE on the test set:          0.906 ($90,607)
Linear regression MAE on the test set: 0.533 ($53,320)
Improvement over the baseline:         0.373 ($37,287, 41% lower)
Linear regression MAE on the training set: 0.529
Lowest and highest predicted price: -1.01 and 11.50 ($100,000s)


## 6. Does a random forest beat linear regression?

Linear regression is held back by its fixed rates. A **random forest** has no rates at all. It is built out of many **decision trees**, so the tree comes first.

### One decision tree: a game of twenty questions

A decision tree prices a neighbourhood by asking a chain of yes/no questions about its features. "Is the median income above $50,000?" If yes, "Is it further north than latitude 37?" and so on. Each answer sends the neighbourhood down a different branch. When the questions run out it has reached a **leaf**, a small group of training neighbourhoods that gave the same answers, and the prediction is the average price of that group.

Nobody writes the questions. During training the tree tries many possible questions and keeps the one that best separates cheap neighbourhoods from expensive ones, then repeats this on each branch.

Because the questions can differ on each branch, a tree is not tied to one rate. It can learn that extra income matters a lot near the coast and less inland, which linear regression cannot.

The weakness is that a single tree keeps asking questions until each leaf holds only a handful of neighbourhoods. By then it has memorised its training rows, quirks included. That is the **overfitting** described in section 3.

### The forest: asking 100 estate agents instead of one

Imagine asking one estate agent to value a house. They are experienced, but their opinion is shaped by the particular houses they happen to have sold. If they once sold an oddly expensive house on a similar street, their estimate will be too high.

Now ask 100 estate agents, each of whom learned the trade from a different mix of past sales, and take the average of their answers. Each agent still has their own quirks, but the quirks are different from agent to agent. One guesses too high, another too low, and in the average those mistakes largely cancel out. What is left is the part they all agree on, which is the real pattern.

A random forest is exactly that:

- Each **tree** is one estate agent.
- Each tree is trained on its own random sample of the training rows. The sample is drawn **with replacement**, meaning a row can be picked more than once and some rows are not picked at all, so every tree sees a slightly different version of the data. This is the "random" in the name.
- To predict, every tree gives its own price and the forest returns the **average** of them.

A model made by combining many smaller models like this is called an **ensemble**.

### What the code does

It is the same create, fit, predict pattern as section 5, with a different model:

- `n_estimators=100` is the number of trees. 100 is scikit-learn's default, written out here so it is visible.
- `random_state=42` fixes the random sampling so the forest comes out the same on every run, for the same reason as in section 3.
- `n_jobs=-1` tells scikit-learn to use every processor core in the computer. The trees are independent of each other, so they can be built at the same time.

The code scores the forest with MAE on the test set, as before, and also on the training set to check for overfitting.

**What to look for:** a test MAE of about 0.328, against 0.533 for linear regression. The training MAE is much lower, about 0.122. The lowest and highest predictions are about 0.49 and 5.00, so the impossible prices from section 5 are gone.

**Finding:** the random forest is wrong by about $32,800 on average, compared with $53,300 for linear regression. That is $20,500 closer, a 39% lower error, and about 16% of a typical neighbourhood's price. Its predictions cannot be negative or go above the cap, because every prediction is an average of real training prices. The gap between the training MAE (0.122) and the test MAE (0.328) shows the trees have partly memorised their training rows, so the test MAE is the honest number to quote, and there may be room to improve the forest by adjusting its settings.

In [6]:
forest = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# Build 100 trees, each on its own random sample of the training set.
forest.fit(X_train, y_train)

# Each tree predicts a price and the forest returns the average.
forest_predictions = forest.predict(X_test)

forest_mae = mean_absolute_error(y_test, forest_predictions)
forest_train_mae = mean_absolute_error(y_train, forest.predict(X_train))
forest_improvement = linear_mae - forest_mae

print(f"Linear regression MAE on the test set: {linear_mae:.3f} "
      f"(${linear_mae * DOLLARS_PER_UNIT:,.0f})")
print(f"Random forest MAE on the test set:     {forest_mae:.3f} "
      f"(${forest_mae * DOLLARS_PER_UNIT:,.0f})")
print(f"Improvement over linear regression:    {forest_improvement:.3f} "
      f"(${forest_improvement * DOLLARS_PER_UNIT:,.0f}, "
      f"{forest_improvement / linear_mae:.0%} lower)")
print(f"Random forest MAE on the training set: {forest_train_mae:.3f}")
print(f"Lowest and highest predicted price: {forest_predictions.min():.2f} "
      f"and {forest_predictions.max():.2f} ($100,000s)")

Linear regression MAE on the test set: 0.533 ($53,320)
Random forest MAE on the test set:     0.328 ($32,784)
Improvement over linear regression:    0.205 ($20,536, 39% lower)
Random forest MAE on the training set: 0.122
Lowest and highest predicted price: 0.49 and 5.00 ($100,000s)


## 7. Which model predicts best?

The three results so far are spread across three sections. Putting them in one table makes them easy to compare. All three were scored the same way, with MAE on the same 4,128 test neighbourhoods, so the comparison is fair.

### What the code does

- `pd.Series(...)` collects the three test MAEs into one labelled column, with the model names as the row labels (the **index**).
- `pd.DataFrame({...})` builds the table from three columns, each calculated from that Series:
  - **Test MAE** is the raw error in units of $100,000, rounded to 3 decimal places.
  - **Error in dollars** multiplies the MAE by 100,000. `.map("${:,.0f}".format)` turns each number into text such as `$32,784`, with a dollar sign, commas and no decimals.
  - **Lower than baseline** is how much of the baseline's error the model removed, as a percentage.
- Ending the cell with `results` on a line by itself makes the notebook display it as a formatted table.

**What to look for:** the error should fall on every row, from $90,607 to $53,320 to $32,784. The baseline is 0% lower than itself, which is a quick check that the last column is calculated correctly.

**Finding:** the random forest is the best of the three, with an average error of about $32,800. That is 64% lower than the baseline, where linear regression managed 41%. Each step up in flexibility, from ignoring the features, to fixed rates, to many trees of questions, cut the error by a large amount.

In [7]:
test_mae = pd.Series(
    {
        "Baseline (average price)": baseline_mae,
        "Linear regression": linear_mae,
        "Random forest": forest_mae,
    }
)

results = pd.DataFrame(
    {
        "Test MAE ($100,000s)": test_mae.round(3),
        "Error in dollars": (test_mae * DOLLARS_PER_UNIT).map("${:,.0f}".format),
        "Lower than baseline": ((baseline_mae - test_mae) / baseline_mae).map(
            "{:.0%}".format
        ),
    }
)

results

,"Test MAE ($100,000s)",Error in dollars,Lower than baseline
Baseline (average price),0.906,"$90,607",0%
Linear regression,0.533,"$53,320",41%
Random forest,0.328,"$32,784",64%


## 8. Is the random forest overfitting?

**Overfitting** is when a model learns its training rows too closely. As well as the general pattern (richer neighbourhoods cost more), it also learns the accidents of those particular rows, which do not repeat in new data. Section 3 compared this to a student who memorises the practice sheet: full marks on the questions they have seen, a lower mark on fresh ones.

The way to detect it is to score the model twice with the same measure:

- The **training MAE** is the error on the rows the model learned from. This is the practice sheet.
- The **test MAE** is the error on rows it has never seen. This is the fresh exam.

If the two are close, the model has learned a pattern that carries over to new data. If the training MAE is much lower than the test MAE, part of what the model learned only works on rows it has already seen. That gap is the sign of overfitting.

### What the code does

It builds a table with one row per model. Both MAEs were already calculated in sections 5 and 6, so nothing is retrained here.

- **Training MAE** and **Test MAE** are the two scores, in units of $100,000.
- **Gap in dollars** is the test MAE minus the training MAE, converted to dollars: how much worse the model does on new neighbourhoods.
- **Test / training** divides one by the other. A value near 1 means the scores match. A value of 2 means the error doubles on unseen data.

The baseline is left out because it only learns one number, so it has nothing to memorise.

**What to look for:** linear regression has almost the same error on both sets (0.529 and 0.533, a ratio of about 1.0). The random forest has a training MAE of about 0.122 and a test MAE of about 0.328, a ratio of about 2.7.

**Finding:** yes, the random forest is overfitting. Its error is about $12,200 on neighbourhoods it trained on and about $32,800 on new ones, 2.7 times higher. The cause is that each tree keeps asking questions until its leaves hold very few neighbourhoods, so the trees have largely memorised their training rows. Linear regression is not overfitting, because 9 numbers (a starting price and 8 rates) are far too few to memorise 16,512 rows. Overfitting does not make the forest the worse model, though: its test MAE of 0.328 still beats linear regression's 0.533, and the test MAE is what counts. It does mean that 0.122 must never be quoted as the forest's accuracy, and that limiting how far the trees grow might lower the test error further.

In [8]:
overfit_check = pd.DataFrame(
    {
        # train_mae is the linear regression's training MAE from section 5.
        "Training MAE": [train_mae, forest_train_mae],
        "Test MAE": [linear_mae, forest_mae],
    },
    index=["Linear regression", "Random forest"],
)

gap = overfit_check["Test MAE"] - overfit_check["Training MAE"]
ratio = overfit_check["Test MAE"] / overfit_check["Training MAE"]

overfit_check["Gap in dollars"] = (gap * DOLLARS_PER_UNIT).map("${:,.0f}".format)
overfit_check["Test / training"] = ratio.round(1)

overfit_check.round(3)

,Training MAE,Test MAE,Gap in dollars,Test / training
Linear regression,0.529,0.533,$457,1.0
Random forest,0.122,0.328,"$20,568",2.7


## 9. What price does the model predict for a made-up neighbourhood?

So far the models have only predicted prices for rows that came with the dataset. This section uses the random forest, the best model from section 7, on details that are typed in by hand. Using a trained model on new input like this is called **inference**.

Each row in the data is a neighbourhood, not a single house, so the details to type in describe a made-up neighbourhood and the answer is its **median house value** in 1990 dollars.

### What to type in

Change the numbers in `my_neighbourhood` and run the cell again. The units must match the ones the model was trained on:

| Name | Meaning | Typical value | Range in the data |
|---|---|---|---|
| `MedInc` | Median household income, in units of $10,000 (5.0 means $50,000) | 3.5 | 0.5 to 15 |
| `HouseAge` | Median age of the houses, in years | 29 | 1 to 52 |
| `AveRooms` | Average number of rooms per household | 5.2 | about 1 to 10 |
| `AveBedrms` | Average number of bedrooms per household | 1.05 | about 0.5 to 2 |
| `Population` | Number of people living in the neighbourhood | 1,166 | 3 to 35,682 |
| `AveOccup` | Average number of people per household | 2.8 | about 1 to 5 |
| `Latitude` | How far north, in degrees | 34.26 | 32.54 to 41.95 |
| `Longitude` | How far west, in degrees (negative in California) | -118.49 | -124.35 to -114.31 |

The starting values describe a neighbourhood in central Los Angeles (latitude 34.05, longitude -118.25). San Francisco is at about 37.77, -122.42.

### What the code does

- `my_neighbourhood` is a **dictionary**: pairs of a name and a value, written inside curly brackets. The names are the 8 feature names.
- `pd.DataFrame([my_neighbourhood])` turns the dictionary into a table with one row. The model was trained on a table, so it expects a table back, even for a single prediction. The square brackets make a list holding one dictionary, and each dictionary in the list becomes one row.
- `[X.columns]` puts the columns in the same order as the training data, so the order they are typed in does not matter. If a name is misspelled or missing, this line stops with a `KeyError` that names the problem column.
- `forest.predict(...)` returns an array with one prediction per row. `[0]` takes the first and only value out of it.
- The last lines print the price in dollars, with a range of one test MAE (from section 6) either side as a rough guide to how far off it could be.

**What to look for:** with the starting values, a predicted price of about 2.312, which is $231,222, and a rough range of $198,438 to $264,006.

**Finding:** the model can price a neighbourhood it has never seen from 8 numbers. Treat the answer as an estimate: the range is the model's *average* error, so some predictions will be further off than that. The prediction can never go above about $500,000 because of the cap on the training prices, and values far outside the ranges in the table give unreliable answers, because the forest has no similar training rows to average.

In [13]:
# Change these numbers, then run the cell again.
my_neighbourhood = {
    "MedInc": 6.0,         # median income in $10,000s (5.0 = $50,000)
    "HouseAge": 15,        # median house age in years (1 to 52)
    "AveRooms": 6.0,       # average rooms per household
    "AveBedrms": 1.1,      # average bedrooms per household
    "Population": 1200,    # people living in the neighbourhood
    "AveOccup": 3.0,       # average people per household
    "Latitude": 34.05,     # degrees north (32.54 to 41.95)
    "Longitude": -118.25,  # degrees west, negative (-124.35 to -114.31)
}

# One-row table with the columns in the same order as the training data.
my_features = pd.DataFrame([my_neighbourhood])[X.columns]

# predict returns one value per row, so [0] takes the only one.
predicted_price = forest.predict(my_features)[0]

low_price = predicted_price - forest_mae
high_price = predicted_price + forest_mae

print(f"Predicted median house value: {predicted_price:.3f} "
      f"(${predicted_price * DOLLARS_PER_UNIT:,.0f})")
print(f"Rough range (one test MAE either side): "
      f"${low_price * DOLLARS_PER_UNIT:,.0f} to "
      f"${high_price * DOLLARS_PER_UNIT:,.0f}")

Predicted median house value: 3.032 ($303,238)
Rough range (one test MAE either side): $270,454 to $336,022
